# 08 — ヒストグラム

## 目標
- `cv2.calcHist` でヒストグラムを計算
- `cv2.equalizeHist` でグローバル均一化
- CLAHE (Contrast Limited Adaptive Histogram Equalization) で局所均一化

## CLAHE のパラメータ
- `clipLimit`: コントラスト上限 (大きいほど強調が強い)
- `tileGridSize`: タイル分割数 (小さいほど局所的)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils import DATA_DIR, load_image, show_nb

src  = load_image(DATA_DIR / 'lena.png')
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

In [ ]:
# matplotlib でヒストグラムを描画
def plot_hist(imgs, labels, title='Histogram'):
    fig, axes = plt.subplots(1, len(imgs), figsize=(5*len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]
    for ax, img, label in zip(axes, imgs, labels):
        hist = cv2.calcHist([img], [0], None, [256], [0, 256])
        ax.plot(hist)
        ax.set_title(label)
        ax.set_xlim([0, 256])
        ax.set_xlabel('pixel value')
        ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()

# 均一化
eq = cv2.equalizeHist(gray)
clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_img = clahe_obj.apply(gray)

show_nb([
    ('gray', gray),
    ('equalizeHist', eq),
    ('CLAHE', clahe_img),
], cols=3)
plot_hist([gray, eq, clahe_img], ['gray', 'equalizeHist', 'CLAHE'])

In [ ]:
# カラー画像の CLAHE: L*a*b* の L チャネルだけに適用
lab = cv2.cvtColor(src, cv2.COLOR_BGR2Lab)
l, a, b = cv2.split(lab)
l_clahe = clahe_obj.apply(l)
result = cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_Lab2BGR)

show_nb([('original', src), ('color CLAHE (Lab L)', result)], cols=2)